## Building A chatbot 

this chatbot that we build will use the language model to have a conversation. There are several other realted concepts that 

    conversational RAG : Enable a chatbot experience over an external source of data

    Agents : Build a chatbot that can take actions.

    

In [51]:
import os 
from dotenv import load_dotenv
load_dotenv() ## loading all the environment variables

groq_api_key=os.environ.get("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model


e:\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A0ACD5EAA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A0ACD5EFE0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi, My name is Anant anf I am chief AI Engineer")])


AIMessage(content="Nice to meet you, Anant. It's great to know that you're a chief AI Engineer. What brings you here today? Do you have a specific project or question regarding AI that you'd like to discuss?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 49, 'total_tokens': 94, 'completion_time': 0.070069663, 'completion_tokens_details': None, 'prompt_time': 0.003857041, 'prompt_tokens_details': None, 'queue_time': 0.159677809, 'total_time': 0.073926704}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5f9b-376b-7213-820d-d5c081991a83-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 45, 'total_tokens': 94})

In [ ]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Anant and I am chief AI Engineer"),
        AIMessage(content="Nice to meet you, Anant. It's great to know that you're a chief AI Engineer. What brings you here today? Do you have a specific project or question regarding AI that you'd like to discuss?"),
        HumanMessage(content="Hey what's my name and profession?")
    ]
)

AIMessage(content='Your name is Anant, and you are the Chief AI Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 111, 'total_tokens': 126, 'completion_time': 0.02514845, 'completion_tokens_details': None, 'prompt_time': 0.008025267, 'prompt_tokens_details': None, 'queue_time': 0.050055443, 'total_time': 0.033173717}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5f9e-9ce5-7b30-8195-b11ada9456e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 15, 'total_tokens': 126})

### Message History
We can use a message History class to wrap our model and make it stateful. This will keep track of inputs and output of the models, and store them in some datastore. Future interactions will then load those message and pass them into the chain as part of the input.

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model,get_session_history)

e:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
config = {"configurable": {"session_id": "chat1"}}

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Anant and I am chief AI Engineer")],
        config=config
)

response.content

"Hello Anant, it seems like we're having a bit of a loop here. Let's start fresh. As the chief AI Engineer, I'm sure you must have a wealth of knowledge and experience in the field. Can you tell me a bit about the work you're currently doing or any exciting projects you're leading?"

In [13]:
with_message_history.invoke(
    [HumanMessage(content="What's my name and profession?")],
    config=config
)

AIMessage(content='Your name is Anant, and your profession is Chief AI Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 292, 'total_tokens': 307, 'completion_time': 0.011380188, 'completion_tokens_details': None, 'prompt_time': 0.018722386, 'prompt_tokens_details': None, 'queue_time': 0.055453074, 'total_time': 0.030102574}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5fb2-f954-7b72-8a6b-b96f332bd326-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 292, 'output_tokens': 15, 'total_tokens': 307})

In [16]:
## changing the config -- session id

config1 = {"configurable": {"session_id": "chat2"}}
response= with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain or access information about individual users. Each time you interact with me, it's a new conversation and I don't retain any context or information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!"

## Prompt Templates 

Prompt Template help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM, Let's now make that a bit more complicated. 
Firstm let's add in a message with some custom instructions(but still taking message as input). Next, we'll add in more input beside just the messages.

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the next of your ability."),
        MessagesPlaceholder(variable_name="messages")

    ]
)

chain = prompt | model 

In [22]:
chain.invoke({"messages":[HumanMessage(content=" Hi My name is Anant")]})

AIMessage(content="Hello Anant, nice to meet you. I'm here to assist you with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 58, 'total_tokens': 93, 'completion_time': 0.04409012, 'completion_tokens_details': None, 'prompt_time': 0.004293691, 'prompt_tokens_details': None, 'queue_time': 0.061320278, 'total_time': 0.048383811}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5fd0-c8ee-7bf3-a537-6502134898a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 35, 'total_tokens': 93})

In [23]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

e:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [24]:
config = {"configurable": {"session_id" : "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Anant")],
    config=config
)

response.content

'Nice to meet you, Anant. Is there something I can help you with today?'

In [25]:
## Adding some more comlexity to the conversation

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",
        "You are a helpful assistant. Answer all the questions to the next of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages")

    ]
)

chain = prompt | model



In [26]:
response=chain.invoke({"messages":[HumanMessage(content="Hi my name is Anant")],"language":"Hindi"})
response.content


'नमस्ते अनंत, मैं आपकी सहायता के लिए यहाँ हूँ। क्या आपके पास कोई प्रश्न या विषय है जिस पर चर्चा करना चाहते हैं?'

Now wrapping this to more complicated cahin in a message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history

In [41]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")


e:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [43]:
config={"configurable": {"session_id" : "chat4"}}

response=with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi, I am Anant")],"language" :"Hindi"},
    config=config
)
response.content

'नमस्ते अनंत, मैं आपकी सहायता के लिए यहाँ हूँ। आप कैसे हैं और मुझे क्या सिखाने या जानने की जरूरत है?'

In [45]:
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="What's my name?")],"language":"Hindi"},
    config=config
)
response.content


'तुम्हारा नाम अनंत है।'

### Managing the conversation history

One important concept to understand when building chatbots is how to manage conversation history. If left unmangedm the list of messages will grow unbounded and potenrially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.


In [46]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=True,
    start_on="human"
)

messages=[
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="Hi, My name is Anant "),
    AIMessage(content="Hi !"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="That's great! "),
    HumanMessage(content="What is 2 * 2?"),
    AIMessage(content="2 * 2 is 4."),
    HumanMessage(content="Thanks!"),
    AIMessage(content="You're welcome!"),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="Paris")

]
trimmer.invoke(messages)


e:\Langchain\venv\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
e:\Langchain\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kumar\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate 

[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 2 * 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='2 * 2 is 4.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Thanks!', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You're welcome!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Paris', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [47]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")| trimmer)
    | prompt
    | model
)

response=chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What is the capital of France?")],
        "language": "English"
    }
)
response.content


'Paris.'

In [48]:
## let's wrap this in the messages history

with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")
config={"configurable": {"session_id" : "chat5"}}



e:\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [49]:
response=with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="What's my name?")],
        "language": "English"
    },
    config=config
)

response.content

"You didn't mention your name, so I don't have any information about it. We just started our conversation."